# SmolVLM-256M: TensorRT Engine for SigLIP

This notebook builds a TensorRT engine for the SigLIP vision encoder. It checks the output against PyTorch, then compares the prefill and the answer.

In [ ]:
!pip install -q tensorrt onnx

In [ ]:
import copy
import gc
import json
import re
import subprocess
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, StaticCache
from transformers.modeling_outputs import BaseModelOutput

ROOT = next(p for p in (Path(".."), Path("."), Path("/content/edge-ai-smol-lvm-250-optimization"))
            if (p / "frame.jpg").exists())
IMAGE = ROOT / "frame.jpg"
MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
PROMPT = "Describe any visible defects."
PREPROCESS = {"default": (None, "cpu"), "slow": (False, "cpu"), "fast-cpu": (True, "cpu"), "fast-gpu": (True, "cuda")}


def timed(fn):
    torch.cuda.synchronize()
    t = time.perf_counter()
    out = fn()
    torch.cuda.synchronize()
    return out, (time.perf_counter() - t) * 1e3


def mean_ms(fn, runs=5):
    fn()
    torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(runs):
        fn()
        torch.cuda.synchronize()
    return (time.perf_counter() - t) * 1e3 / runs


def until_eos(ids, eos):
    for i, t in enumerate(ids):
        if t in eos:
            return ids[:i]
    return ids


def set_attn(model, impl):
    if hasattr(model, "set_attn_implementation"):
        model.set_attn_implementation(impl)
    else:
        for c in (model.config.text_config, model.config.vision_config):
            c._attn_implementation = impl


def load(attn=None, fast=None):
    kw = {} if attn is None else {"attn_implementation": attn}
    proc = AutoProcessor.from_pretrained(MODEL_ID, **({} if fast is None else {"use_fast": fast}))
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float16, **kw).to("cuda").eval()
    return model, proc


def eos_ids(tok):
    return {tok.eos_token_id, tok.convert_tokens_to_ids("<end_of_utterance>")}


def chat_text(proc, prompt):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    return proc.apply_chat_template(msgs, add_generation_prompt=True)


def make_inputs(proc, image, prompt, image_device="cpu", size=None, extra=None):
    img = Image.open(image).convert("RGB") if isinstance(image, (str, Path)) else image
    dev = {} if image_device == "cpu" else {"device": image_device}
    ik = {**({} if size is None else {"size": {"longest_edge": size}}), **dev, **(extra or {})}
    text = chat_text(proc, prompt)
    try:
        out = proc(text=text, images=[img], return_tensors="pt", images_kwargs=ik)
    except (TypeError, ValueError):
        out = proc(text=text, images=[img], return_tensors="pt", **ik)
    out = out.to("cuda")
    out["pixel_values"] = out["pixel_values"].to(torch.float16)
    return out


def bare_generate(model, inputs, k, stop=True):
    extra = {} if stop else {"min_new_tokens": k}
    with torch.inference_mode():
        return model.generate(**inputs, max_new_tokens=k, do_sample=False, **extra)


class GraphDecoder:
    def __init__(self, model, max_new=64, max_len=4096, own_mask=False):
        self.model, self.max_new, self.max_len = model, max_new, max_len
        self.cache = StaticCache(config=model.config.text_config, max_cache_len=max_len)
        dev = model.device
        self.ids = torch.zeros(1, 1, dtype=torch.long, device=dev)
        self.pos = torch.zeros(1, dtype=torch.long, device=dev)
        self.idx = torch.zeros(1, dtype=torch.long, device=dev)
        self.out = torch.zeros(max_new, dtype=torch.long, device=dev)
        self.graph = None
        self.steps = max_new
        self.own_mask = own_mask or model.config.text_config._attn_implementation == "eager"

    def mask(self):
        allowed = torch.arange(self.max_len, device=self.pos.device) <= self.pos[:, None]
        dt = self.model.dtype
        return ((~allowed).to(dt) * torch.finfo(dt).min)[:, None, None, :]

    def step(self):
        extra = {"attention_mask": self.mask()} if self.own_mask else {}
        h = self.model.model.text_model(
            input_ids=self.ids, cache_position=self.pos, past_key_values=self.cache, use_cache=True, **extra
        ).last_hidden_state
        nxt = self.model.lm_head(h[:, -1]).argmax(-1, keepdim=True)
        self.ids.copy_(nxt)
        self.out.index_copy_(0, self.idx, nxt.view(1))
        self.pos.add_(1)
        self.idx.add_(1)

    @torch.inference_mode()
    def capture(self):
        s = torch.cuda.Stream()
        s.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(s):
            for _ in range(3):
                self.pos.fill_(self.max_len - 1)
                self.idx.zero_()
                self.step()
        torch.cuda.current_stream().wait_stream(s)
        self.pos.fill_(self.max_len - 1)
        self.idx.zero_()
        self.graph = torch.cuda.CUDAGraph()
        with torch.cuda.graph(self.graph):
            self.step()

    @torch.inference_mode()
    def prefill(self, inputs):
        self.cache.reset()
        n = inputs["input_ids"].shape[1]
        logits = self.model(
            **inputs, past_key_values=self.cache, use_cache=True,
            cache_position=torch.arange(n, device=self.model.device),
        ).logits
        first = logits[:, -1].argmax(-1, keepdim=True)
        self.ids.copy_(first)
        self.pos.fill_(n)
        self.idx.zero_()
        return first

    @torch.inference_mode()
    def decode(self, use_graph=True, eos=None):
        self.steps = self.max_new
        for i in range(self.max_new):
            if use_graph:
                self.graph.replay()
            else:
                self.step()
            if eos and self.ids.item() in eos:
                self.steps = i + 1
                break
        return self.out.clone()

    def generate(self, inputs, eos, use_graph=True):
        first = self.prefill(inputs)
        ids = torch.cat([first.view(1), self.decode(use_graph, eos)]).tolist()
        return until_eos(ids, eos)


def time_decode(dec, inputs, eos, runs=5, use_graph=True, stop=False):
    pre, dec_ms, ids = [], [], None
    for i in range(runs + 1):
        first, p = timed(lambda: dec.prefill(inputs))
        out, d = timed(lambda: dec.decode(use_graph, eos if stop else None))
        if i:
            pre.append(p)
            dec_ms.append(d)
        ids = until_eos(torch.cat([first.view(1), out]).tolist(), eos)
    return {"prefill_ms": float(np.mean(pre)), "decode_ms": float(np.mean(dec_ms)), "steps": dec.steps, "ids": ids}


def fit_len(n, max_new):
    return -(-(n + max_new + 1) // 64) * 64

In [ ]:
TRT = ROOT / "trt_siglip"


class SiglipTrt(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.vision = model.model.vision_model
        self.owner = [model]
        self.engines = {}

    def forward(self, pixel_values, patch_attention_mask=None, *args, **kwargs):
        if patch_attention_mask is not None and not bool(patch_attention_mask.all()):
            return self.vision(pixel_values, patch_attention_mask, *args, **kwargs)
        hidden = self.vision.embeddings(pixel_values, patch_attention_mask)
        n = hidden.shape[0]
        if n not in self.engines:
            self.engines[n] = engine_for(self.owner[0], self.vision, hidden)
        return BaseModelOutput(last_hidden_state=self.engines[n](hidden))


class SiglipEncoder(torch.nn.Module):
    def __init__(self, vision):
        super().__init__()
        self.encoder = vision.encoder
        self.norm = vision.post_layernorm

    def forward(self, x):
        return self.norm(self.encoder(inputs_embeds=x)[0])


def export_onnx(model, vision, x, path):
    set_attn(model, "eager")
    enc = copy.deepcopy(SiglipEncoder(vision)).half().eval()
    set_attn(model, "sdpa")
    with torch.no_grad():
        torch.onnx.export(enc, (x.half(),), str(path), dynamo=False,
                          input_names=["embeds"], output_names=["hidden"], opset_version=17)


def build_engine(onnx, engine):
    import tensorrt as trt

    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    nf = trt.NetworkDefinitionCreationFlag
    has_fp16 = hasattr(trt.BuilderFlag, "FP16")
    flag = 1 << int(nf.EXPLICIT_BATCH) if hasattr(nf, "EXPLICIT_BATCH") else 0
    if not has_fp16:
        flag |= 1 << int(nf.STRONGLY_TYPED)
    network = builder.create_network(flag)
    parser = trt.OnnxParser(network, logger)
    if not parser.parse(Path(onnx).read_bytes()):
        raise RuntimeError("; ".join(str(parser.get_error(i)) for i in range(parser.num_errors)))
    config = builder.create_builder_config()
    if has_fp16:
        config.set_flag(trt.BuilderFlag.FP16)
    data = builder.build_serialized_network(network, config)
    if data is None:
        raise RuntimeError("TensorRT build failed")
    Path(engine).write_bytes(data)


class TrtEngine:
    def __init__(self, path):
        import tensorrt as trt

        engine = trt.Runtime(trt.Logger(trt.Logger.WARNING)).deserialize_cuda_engine(Path(path).read_bytes())
        self.ctx = engine.create_execution_context()
        names = [engine.get_tensor_name(i) for i in range(engine.num_io_tensors)]
        self.inp = next(n for n in names if engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT)
        self.out = next(n for n in names if engine.get_tensor_mode(n) == trt.TensorIOMode.OUTPUT)
        self.out_shape = tuple(engine.get_tensor_shape(self.out))
        dtypes = {trt.DataType.FLOAT: torch.float32, trt.DataType.HALF: torch.float16}
        self.in_dtype = dtypes[engine.get_tensor_dtype(self.inp)]
        self.out_dtype = dtypes[engine.get_tensor_dtype(self.out)]

    def __call__(self, x):
        x = x.to(self.in_dtype).contiguous()
        y = torch.empty(self.out_shape, dtype=self.out_dtype, device=x.device)
        self.ctx.set_tensor_address(self.inp, x.data_ptr())
        self.ctx.set_tensor_address(self.out, y.data_ptr())
        self.ctx.execute_async_v3(torch.cuda.current_stream().cuda_stream)
        return y.half()


def engine_for(model, vision, hidden):
    TRT.mkdir(parents=True, exist_ok=True)
    n = hidden.shape[0]
    onnx, engine = TRT / f"siglip_{n}tiles.onnx", TRT / f"siglip_{n}tiles.engine"
    if not engine.exists():
        with torch.inference_mode(False):
            export_onnx(model, vision, hidden.clone(), onnx)
            build_engine(onnx, engine)
    return TrtEngine(engine)

In [ ]:
model, proc = load()
inputs = make_inputs(proc, IMAGE, PROMPT)
eos = eos_ids(proc.tokenizer)
captured = {}


def grab(module, args, kwargs):
    captured["x"] = (kwargs["inputs_embeds"] if "inputs_embeds" in kwargs else args[0]).detach().clone()


handle = model.model.vision_model.encoder.register_forward_pre_hook(grab, with_kwargs=True)
with torch.no_grad():
    model(**inputs, use_cache=False)
handle.remove()
x = captured["x"]
vision = model.model.vision_model
engine = engine_for(model, vision, x)
with torch.no_grad():
    ref = vision.post_layernorm(vision.encoder(inputs_embeds=x)[0]).float()
diff = (engine(x).float() - ref).abs()
print(f"output vs PyTorch: max {diff.max():.4f} | mean {diff.mean():.5f} | mean value {ref.abs().mean():.3f}")

dec = GraphDecoder(model, 64, 1024)
dec.capture()
pt = time_decode(dec, inputs, eos, stop=True)
pt_siglip = mean_ms(lambda: vision(pixel_values=inputs["pixel_values"], patch_attention_mask=inputs.get("pixel_attention_mask")))
model.model.vision_model = SiglipTrt(model)
trt = time_decode(dec, inputs, eos, stop=True)
trt_siglip = mean_ms(lambda: model.model.vision_model(pixel_values=inputs["pixel_values"], patch_attention_mask=inputs.get("pixel_attention_mask")))
pd.DataFrame([
    {"option": "pytorch", "prefill ms": round(pt["prefill_ms"], 1), "SigLIP ms": round(pt_siglip, 1),
     "same answer": True},
    {"option": "tensorrt", "prefill ms": round(trt["prefill_ms"], 1), "SigLIP ms": round(trt_siglip, 1),
     "same answer": trt["ids"] == pt["ids"]},
])

## Log of previous runs

The cells below are the earlier notebook with their outputs. They call the scripts that were replaced by the code above, so they are not run again.

In [ ]:
# Colab only. The Jetson has TensorRT in JetPack, but it may need the onnx package.
!pip install -q tensorrt onnx

In [ ]:
REPO = "./edge-ai-smol-lvm-250-optimization"
!python {REPO}/trt_siglip.py --image {REPO}/frame.jpg --max-len 1024